### RAG Evaluation

##### Loading the RAG answers

In [51]:
import pandas as pd

df_answers = pd.read_csv("rag-answers-new.csv")
answers = df_answers.to_dict(orient="records")

In [52]:
answers

[{'question': 'Is it okay to join the course late if I just found it now?',
  'answer_llm': 'Yes, you can still join the course late. If you want a certificate, though, you need to submit your project while submissions are still being accepted.',
  'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'document': '74eb249bbf'},
 {'question': 'Can I still take this course even if I missed the start date?',
  'answer_llm': 'Yes, you can still join if you missed the start date, but if you want a certificate, you need to finish with the live cohort and submit your project while submissions are still open.',
  'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'document': '74eb249bbf'},
 {'question': 'If I join after the course has already started, am I still eligible for a certificate?',
  'answer_llm': 'Yes, as long 

#### A->Q->A' evaluation

In [11]:
from pydantic import BaseModel, Field
from typing import Literal

class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer."
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )

In [12]:
# First, write the judge instructions. This tells the judge what to compare and how to assign the score.
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

In [53]:
# Then define the prompt template. This is the data we pass to the judge for each answer.
aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

In [ ]:
import os

In [ ]:
from openai import OpenAI

# Resolve API key from environment if available (OPENAI_API_KEY preferred, fallback to GROQ_API_KEY)
api_key = os.getenv("OPENAI_API_KEY") or os.getenv("GROQ_API_KEY")
if not api_key:
    raise RuntimeError("Set OPENAI_API_KEY or GROQ_API_KEY environment variable before creating OpenAI client.")
openai_client = OpenAI(
    api_key=api_key,
    base_url=os.getenv("GROQ_BASE_URL", "https://api.groq.com/openai/v1") or os.getenv("OPENAI_BASE_URL", "https://api.openai.com/v1")
)

In [56]:
# Import the structured-output helper:
from evaluation_utils import calc_price, calc_total_price, llm_structured_retry, map_progress

In [58]:
# Take one record:
rec = answers[0]

In [48]:
rec

{'question': 'Is it okay to join the course late if I just found it now?',
 'answer_llm': 'Yes, you can still join the course late. If you want a certificate, though, you need to submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [59]:
prompt = aqa_judge_prompt.format(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

In [60]:
# Call the judge:
eval_result, usage = llm_structured_retry(
    openai_client,
    aqa_judge_instructions,
    prompt,
    AnswerEvaluation,
    model="openai/gpt-oss-20b",
)

eval_result

AnswerEvaluation(reasoning='The AI answer conveys the same core information as the original: it confirms that joining late is allowed and emphasizes that a certificate requires a project submission while submissions are still being accepted. No key point is omitted or misrepresented.', score='good')

In [61]:
# Check the cost:
calc_price(usage)

{'input_cost': 0.00032175, 'output_cost': 0.0009045, 'total_cost': 0.00122625}

In [62]:
# Now put the same logic into a function:
def evaluate_aqa(question, answer_orig, answer_llm, model="gpt-5.4-mini"):
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm
    )

    result, usage = llm_structured_retry(
        openai_client,
        aqa_judge_instructions,
        prompt,
        AnswerEvaluation,
        model=model,
    )

    return result, usage

In [63]:
# Test it on the same record:
eval_result, usage = evaluate_aqa(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"],
    model="openai/gpt-oss-20b",

)

eval_result

AnswerEvaluation(reasoning='The AI answer accurately reflects the key information of the original answer: it confirms that late enrollment is allowed and states that a certificate can only be obtained if the project is submitted while submissions are still being accepted. No essential detail is omitted, and the meaning is preserved.', score='good')

#### Running the judge

In [64]:
def judge_record(rec):
    eval_result, usage = evaluate_aqa(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_llm=rec["answer_llm"],
        model="openai/gpt-oss-20b",
    )

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "score": eval_result.score,
        "reasoning": eval_result.reasoning,
    }

    return result, usage

In [ ]:
# Use the same parallel processing helper:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, answers[:10], judge_record)

  0%|          | 0/10 [00:00<?, ?it/s]

In [66]:
# Split the results:
evaluations = []
usages = []

for evaluation, usage in results:
    evaluations.append(evaluation)
    usages.append(usage)

In [70]:
evaluations

[{'question': 'Is it okay to join the course late if I just found it now?',
  'document': '74eb249bbf',
  'score': 'good',
  'reasoning': 'The AI answer accurately conveys the same information as the original: it affirms that joining late is permissible, and it states that a certificate requires the project to be submitted while submissions are still accepted. No key points are omitted or misstated.'},
 {'question': 'Can I still take this course even if I missed the start date?',
  'document': '74eb249bbf',
  'score': 'good',
  'reasoning': 'The AI answer correctly states that a student can still join the course after missing the start date and that a certificate requires submitting the project while submissions are still open. The added detail about finishing with the live cohort is not present in the ground truth, but it does not contradict or omit the essential information and can be considered additional context. Thus, the AI answer is semantically equivalent to the original.'},
 {

In [69]:
# Create a dataframe:
df_eval = pd.DataFrame(evaluations)
df_eval

,question,document,score,reasoning
0,Is it okay to join the course late if I just f...,74eb249bbf,good,The AI answer accurately conveys the same info...
1,Can I still take this course even if I missed ...,74eb249bbf,good,The AI answer correctly states that a student ...
2,If I join after the course has already started...,74eb249bbf,good,The AI answer correctly states that a certific...
3,Do I need to submit my project before submissi...,74eb249bbf,good,The AI answer correctly states that you must s...
4,I’m a bit late to the course—what do I need to...,74eb249bbf,bad,The original answer states only that the stude...
5,I registered for the LLM Zoomcamp — when shoul...,977bf7786c,good,The AI answer captures the essential informati...
6,Do I actually need an acceptance email before ...,977bf7786c,good,The AI answer contains all key information: it...
7,"If I filled out the registration form, does th...",977bf7786c,good,The AI answer captures all the essential point...
8,"Is registering for the LLM Zoomcamp required, ...",977bf7786c,good,The AI answer accurately states that registrat...
9,Can I begin learning and submit homework even ...,977bf7786c,good,The AI answer conveys the essential points fro...


In [75]:
df_eval.score.value_counts(normalize=True)

,proportion
score,
good,0.9
bad,0.1


In [71]:
# Check the results:
good_count = (df_eval["score"] == "good").sum()
total_count = len(df_eval)
print(f"Good: {good_count}/{total_count} = {good_count/total_count:.2%}")

Good: 9/10 = 90.00%


In [72]:
# Look at the "bad" cases to understand what went wrong:
df_eval[df_eval["score"] == "bad"].head()

,question,document,score,reasoning
4,I’m a bit late to the course—what do I need to...,74eb249bbf,bad,The original answer states only that the stude...


In [74]:
df_eval.to_csv("rag-evaluations-new.csv", index=False)